# MiniCPM5-1B fine-tune on Modal Notebooks

Interactive path for the **Modal** + **Well-Tuned** hackathon tracks.

**Setup (sidebar before running cells):**
1. [modal.com/notebooks](https://modal.com/notebooks) — upload this `.ipynb`
2. **Compute profile** → enable GPU (e.g. A10G)
3. **Files** → attach Volume `slm-finetune` (mounts at `/mnt/slm-finetune`)
4. **Secrets** → attach `huggingface` (`HF_TOKEN`)

Model: [openbmb/MiniCPM5-1B](https://huggingface.co/openbmb/MiniCPM5-1B)

Docs: [Modal Notebooks](https://modal.com/docs/guide/notebooks) · [Volumes](https://modal.com/docs/guide/volumes)

For reproducible sweeps: `modal run research/modal/finetune_app.py`

In [ ]:
# Verify GPU (Modal Notebooks provide CUDA)
!nvidia-smi

In [ ]:
# Clone repo (replace with your fork URL) or upload files via the Files panel
!git clone https://github.com/YOUR_USER/small-model-hackathon.git /root/repo 2>/dev/null || true
%cd /root/repo
!pwd && ls research/finetune.py models.yaml

In [ ]:
# Install project deps (Modal default image has torch/transformers; add finetune stack)
%uv pip install uv peft datasets bitsandbytes accelerate
!uv sync --frozen --package inference --package slm-evals --group finetune --group lm-eval --no-dev

In [ ]:
import os
from pathlib import Path

os.environ["TRUST_REMOTE_CODE"] = "true"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

# Persist on attached Volume (ephemeral container disk is lost on kernel stop)
VOL = Path("/mnt/slm-finetune")
OUT = VOL / "lesson-lora-notebook" if VOL.is_dir() else Path("./models/finetuned/minicpm5-1b-lesson-lora")
OUT.mkdir(parents=True, exist_ok=True)
print(f"Checkpoint dir: {OUT}")

## Smoke fine-tune (LoRA, 20 steps)

Uses the lesson-agent chat dataset by default.

In [ ]:
!uv run python research/finetune.py \
  --preset minicpm5-1b \
  --mode lora \
  --dataset research/data/education-lesson-chat.jsonl \
  --format chat \
  --out {OUT} \
  --max_steps 20

## Baseline lm-eval (smoke)

In [ ]:
!uv run --package slm-evals slm-lm-eval \
  --config research/evals/configs/lm_eval_smoke.yaml \
  --preset minicpm5-1b \
  --experiment-name minicpm5-1b__notebook-baseline

## Post-train lm-eval (adapter)

In [ ]:
!uv run --package slm-evals slm-lm-eval \
  --config research/evals/configs/lm_eval_smoke.yaml \
  --model openbmb/MiniCPM5-1B \
  --adapter {OUT} \
  --experiment-name minicpm5-1b-lesson-lora__notebook \
  --compare-to results/lm_eval/minicpm5-1b__notebook-baseline/results.json

## Sample generation

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

base_id = "openbmb/MiniCPM5-1B"
adapter_dir = str(OUT)

tokenizer = AutoTokenizer.from_pretrained(base_id, trust_remote_code=True)
base = AutoModelForCausalLM.from_pretrained(
    base_id, torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True
)
model = PeftModel.from_pretrained(base, adapter_dir)
model.eval()

prompt = "Explain photosynthesis in one short paragraph for a 10-year-old."
if tokenizer.chat_template:
    text = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=False,
        add_generation_prompt=True,
    )
else:
    text = prompt

ids = tokenizer(text, return_tensors="pt").to(model.device)
out = model.generate(**ids, max_new_tokens=120, do_sample=True, temperature=0.7)
print(tokenizer.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True))

## After training

- **Volume attached?** Use the Files panel (⬇) or run locally: `modal volume get slm-finetune lesson-lora-notebook ./models/finetuned/minicpm5-1b-lora`
- **Hub:** `huggingface-cli upload your-user/minicpm5-1b-lesson-lora <path-to-OUT> . --repo-type model`
- **Share notebook:** Share → public link → "Can view and run" for hackathon judges

Full docs: `research/modal/README.md` in the repo.